<a href="https://colab.research.google.com/github/sd9616/PromptDrift/blob/main/scripts/patch_fix/patch_suggestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROMPT DRIFT - PATCH SUGGESTION**

To use Groq, you'll need an API key. You can get one from the [Groq website](https://groq.com/). Once you have your API key, add it to the Colab secrets manager under the "🔑" in the left panel. Name it `GROQ_API_KEY`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get update
!apt-get install -y openjdk-11-jdk

In [ ]:
!update-alternatives --install /usr/bin/java java /usr/lib/jvm/java-11-openjdk-amd64/bin/java 1100
!update-alternatives --install /usr/bin/javac javac /usr/lib/jvm/java-11-openjdk-amd64/bin/javac 1100

In [ ]:
!update-alternatives --config java
!update-alternatives --config javac

In [ ]:
!java -version

In [ ]:
import os

# Choose a folder to store Defects4J projects
PROJECTS_DIR = "/content/drive/MyDrive/defects4j_projects"
os.makedirs(PROJECTS_DIR, exist_ok=True)

!git clone https://github.com/rjust/defects4j.git /content/defects4j

In [ ]:
!apt-get install -y cpanminus

!cpanm --installdeps .

!cpanm String::Interpolate DBI

In [ ]:
os.environ['D4J_HOME'] = "/content/defects4j"
os.environ['PATH'] = os.environ['D4J_HOME'] + "/framework/bin:" + os.environ['PATH']

In [ ]:
!cd $D4J_HOME && ./init.sh

In [ ]:
!defects4j info

In [ ]:
!pip3 install groq

In [ ]:
shared_folder = input()
prompt_dataset_path = input()

In [ ]:
import pandas as pd

df = pd.read_csv(prompt_dataset_path)

# Check the first few rows
print(df.head())

In [ ]:
from groq import Groq
from google.colab import userdata

# Retrieve the API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Initialize the Groq client
client = Groq(
    api_key=GROQ_API_KEY,
)

Now you can use the client to generate content. Let's try with the `llama-3.1-8b-instant` model.

In [ ]:
def generate_code(prompt):
  completion = client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[
          {
              "role": "user",
              "content": prompt
          }
      ]
  )

  return completion.choices[0].message.content

In [ ]:
import os, subprocess

def checkout(project, bug, workdir):
  if os.path.exists(workdir):
      subprocess.run(["rm", "-rf", workdir])

  subprocess.run(
      ["defects4j", "checkout", "-p", project, "-v", bug + "b", "-w", workdir],
      check=True
  )

In [ ]:
import re

def get_patched_files(raw_output):
  pattern = r'---FILE_START---\nFILENAME: (.+?)\nCONTENT:\n(.*?)\n---FILE_END---'
  matches = re.findall(pattern, raw_output, re.DOTALL)

  patched_files = {file_name.strip(): content for file_name, content in matches}
  return patched_files

In [ ]:
def run(patched_files, workdir):
  for file_name, content in patched_files.items():
      package_path = file_name.rsplit(".", 2)[0]  # remove .java
      class_name = file_name.split(".")[-2] + ".java"  # last part + .java
      src_main_java = os.path.join(workdir, "src", "main", "java")
      src_java = os.path.join(workdir, "src", "java")
      src = os.path.join(workdir, "src")

      if os.path.isdir(src_main_java):
          source_root = src_main_java
      elif os.path.isdir(src_java):
          source_root = src_java
      elif os.path.isdir(src):
          source_root = src
      else:
          raise ValueError("No source directory found")

      java_file = os.path.join(
          source_root,
          *package_path.split("."),
          class_name
      )

      os.makedirs(os.path.dirname(java_file), exist_ok=True)
      print(f"Writting {java_file}")

      with open(java_file, "w") as f:
          f.write(content)


  compile_result = subprocess.run(["defects4j", "compile"], cwd=workdir, capture_output=True, text=True)
  test_result = subprocess.run(["defects4j", "test"], cwd=workdir, capture_output=True, text=True)


  if compile_result.returncode != 0:
    return {"compile_status": "failed", "compile_response": compile_result}
  else:
    failed_tests = []
    capture = False

    for line in test_result.stdout.splitlines():
        line = line.strip()
        if line.startswith("Failing tests:"):
            capture = True
            continue
        if capture:
            if line.startswith("- "):
                failed_tests.append(line[2:].strip())  # remove "- " prefix
            else:
                # stop capturing if line doesn't start with "- "
                break

    return {
        "compile_status": "success",
        "failed_tests": failed_tests,
        "compile_respose": compile_result,
        "test_response": test_result
        }


In [ ]:
import csv
import os

# File paths
prompt_csv_path = os.path.join(shared_folder, f"{prompt_dataset_path}_results_run_1.csv")
modified_csv_path = os.path.join(shared_folder, f"{prompt_dataset_path}_modified_files_run_1.csv")
failed_csv_path = os.path.join(shared_folder, f"{prompt_dataset_path}_failed_test_results_run_1.csv")

# Initialize lists (or load existing CSV if it exists)
prompt_results = []
modified_files_dataset = []
failed_test_results = []

In [ ]:
from datetime import datetime
import time

for idx, row in df.iloc[210:].iterrows():
    try:
      project = row["project"]
      bug = str(row["bug_id"])
      prompt = row["prompt"]

      if len(prompt) > 10000 or len(prompt) == 0:
        continue

      print(f"Processing {idx} {project} {bug}")

      workdir = "/tmpbp/" + project + "_" + bug

      checkout(project, bug, workdir)

      raw_output = generate_code(prompt)
      # print(f"AI output {raw_output}")

      patched_files = get_patched_files(raw_output)
      # print(f"Patched files {patched_files}")

      result = run(patched_files, workdir)
      print(f"Result {result}")

      prompt_results = (project, bug, prompt, raw_output, "", result.get("compile_status", ""), len(result.get("failed_tests", [])), ";".join(result.get("failed_tests", [])), datetime.now().isoformat())

      for file_name, content in patched_files.items():
        modified_files_dataset.append((project, bug, file_name, content, datetime.now().isoformat()))

      for failed_test in result.get("failed_tests", []):
        failed_test_results.append((project, bug, failed_test, datetime.now().isoformat()))

      print(f"Saved results for {idx} {project} {bug}")
    except Exception as e:
      print("error occured", e)
      prompt_results = (project, bug, prompt, raw_output if raw_output else "", str(e), result.get("compile_status", ""), len(result.get("failed_tests", [])), ";".join(result.get("failed_tests", [])), datetime.now().isoformat())
      continue

    time.sleep(10)
    # --- Save prompt results to Drive ---
    with open(prompt_csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(prompt_results)

    # --- Save modified files to Drive ---
    with open(modified_csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(modified_files_dataset)

      # --- Save failed tests to Drive ---
    with open(failed_csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(failed_test_results)

